In [26]:
import os
import requests
import pandas as pd
import os
import logging
import time
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [27]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [28]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [29]:
import vertexai
from vertexai import generative_models as genai
from vertexai.generative_models import GenerationConfig

# Model definition
multimodal_model = genai.GenerativeModel( "gemini-1.5-flash",
generation_config= GenerationConfig(temperature=0.7))

In [19]:
def generate_consultas(df, num_consultas=100):
    # Ejemplos de consultas para el modelo
    ejemplos_consultas = [
        "Busco una crema para las estrías",
        "Necesito un suplemento de vitamina D para personas mayores",
        "¿Tienes algún producto para la caída del cabello?",
        "Quiero una crema hidratante para piel sensible",
        "¿Hay algún spray nasal para alergias?"
    ]
    
    # Prompt para el modelo
    prompt = f"""Actúa como un profesional de farmacia y genera UNA ÚNICA consulta de un cliente que busca un producto de parafarmacia.
    La consulta debe ser similar a estos ejemplos: {ejemplos_consultas}
    IMPORTANTE: Genera solo UNA consulta simple, sin categorías, títulos ni formato adicional.
    NO agregues viñetas, números ni otros caracteres especiales.
    el output debe contener solo la consulta."""

    # Generar consultas
    consultas_generadas = []
    for _ in range(num_consultas):
        try:
            response = multimodal_model.generate_content(prompt)
            consulta = response.text.strip()  # Eliminar espacios en blanco extras
            consultas_generadas.append(consulta)
        except Exception as e:
            print(f"Error al generar la consulta: {str(e)}")
            consultas_generadas.append("Error en la generación")

    # Crear un DataFrame con las consultas generadas
    df_consultas = pd.DataFrame({'prompt': consultas_generadas})

    return df_consultas

# Uso de la función
df = generate_consultas(df)
df

,consultas
0,¿Tienen algún producto para aliviar la acidez ...
1,¿Tienen algún producto para la acidez estomacal?
2,¿Tienen algún producto para el dolor de garganta?
3,¿Tienen alguna crema para aliviar la picazón e...
4,Necesito un protector solar para piel sensible.
...,...
95,¿Tienen algún producto para aliviar el dolor d...
96,¿Tienen algún producto para aliviar el dolor d...
97,¿Tienen algún producto para aliviar el dolor d...
98,¿Tienen algún desodorante natural para pieles ...


In [30]:
#Prueba con productos Cofares

def get_products_cofares(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_gecko`,  -- Añadidos los backticks
          (SELECT @prompt AS content),
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      d.es_marca_propia AS marca,
      d.nombre_matricula_nivel0 AS matricula0,
      d.nombre_matricula_nivel1 AS matricula1,
      d.txt_composicion AS composicion,
      d.forma,
      d.color,
      d.descripcion_visual,
      d.empaque,
      d.zona_de_aplicacion,
      ML.DISTANCE(
        qe.query_embedding,
        d.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_and_embeddings` as d
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
      WHERE d.es_marca_propia = TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """
    
    # Imprimir la consulta SQL generada para depuración
    #print(query)  # Esto te ayudará a verificar la consulta

    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:
        # Asignación de valores con lógica adicional
        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'

        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')

        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query,
        })
    return products

In [33]:
def process_consultas_with_bigquery(df):
    # Configuración del proyecto
    PROJECT_ID = "dataton-2024-team-01-cofares"
    
    # Inicializa el cliente de BigQuery con el proyecto específico
    client = bigquery.Client(project=PROJECT_ID)
    
    # Asegurarse de que existe la columna 'bigquery'
    if 'bigquery' not in df.columns:
        df['bigquery'] = None
    
    # Procesar cada consulta en el DataFrame
    for index, row in df.iterrows():
        try:
            # Obtener la consulta de la fila actual
            prompt = row['prompt']
            
            # Obtener los productos para esta consulta
            productos = get_products_cofares(prompt)
            
            # Si hay productos, tomar el primero (el más relevante)
            if productos:
                # Crear un string con la información relevante del producto
                producto_info = {
                    'nombre': productos[0]['nombre'],
                    'codigo_nacional': productos[0]['codigo_nacional'],
                    'descripcion': productos[0]['descripcion'],
                    'distance_to_query': productos[0]['distance_to_query']
                }
                
                # Guardar la información en la columna 'bigquery'
                df.at[index, 'bigquery'] = str(producto_info)
            else:
                df.at[index, 'bigquery'] = "No se encontraron productos"
                
            # Imprimir progreso cada 10 consultas
            if (index + 1) % 10 == 0:
                print(f"Procesadas {index + 1} consultas...")
                
        except Exception as e:
            print(f"Error procesando consulta {index}: {str(e)}")
            df.at[index, 'bigquery'] = f"Error: {str(e)}"
    
    return df

# Uso de la función
df = process_consultas_with_bigquery(df)
print("Procesamiento completado")

Error procesando consulta 0: Project was not passed and could not be determined from the environment.
Error procesando consulta 1: Project was not passed and could not be determined from the environment.
Error procesando consulta 2: Project was not passed and could not be determined from the environment.
Error procesando consulta 3: Project was not passed and could not be determined from the environment.
Error procesando consulta 4: Project was not passed and could not be determined from the environment.
Error procesando consulta 5: Project was not passed and could not be determined from the environment.
Error procesando consulta 6: Project was not passed and could not be determined from the environment.
Error procesando consulta 7: Project was not passed and could not be determined from the environment.
Error procesando consulta 8: Project was not passed and could not be determined from the environment.
Error procesando consulta 9: Project was not passed and could not be determined fr